# Steam Trajectory — Data Exploration

Loading the fronkongames/steam-games-dataset **JSON** export (switched from CSV after validation revealed row-level misalignment — see notes in `kaggle_loader.py`). This notebook confirms the JSON loads cleanly, then runs cohort selection.

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("/Users/pmacias/Dropbox/steamproject")
print(os.getcwd())

/Users/pmacias/Dropbox/steamproject


In [2]:
from steam_trajectory.ingest.kaggle_loader import KaggleLoader

# NOTE: adjust filename below to match whatever the JSON actually
# downloaded as (likely games.json — confirm in data/raw/ first)
loader = KaggleLoader("data/raw/games.json")
df = loader.df
print(df.shape)
df.head()

(135043, 9)


,AppID,Name,Release date,Developers,Publishers,Price,total_reviews,review_score_percent,Genres
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",NaN,NaN,0.00,0,NaN,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",minori,MangaGamer,5.24,255,98.823529,Adventure
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",Somer Games,8floor,4.99,24,87.500000,Casual
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",유진게임즈,유진게임즈,8.99,0,NaN,"Casual, Indie, Simulation"
4,3631080,Maze Quest VR,"Apr 24, 2025",Reality Expanded LLC,Reality Expanded LLC,4.99,0,NaN,"Action, Early Access"


## Validation
Confirming the JSON loaded cleanly before trusting `select_cohort()`'s output. No CSV-style alignment issues are possible here (JSON fields are explicitly keyed, not positional), but worth still checking for data-quality issues (missing dates, bad AppIDs, etc.) independent of the format itself.

In [3]:
import pandas as pd

# 1. Do release dates parse cleanly?
release_dates = pd.to_datetime(df["Release date"], errors="coerce")
print("Unparseable release dates:", release_dates.isna().sum(), "of", len(df))

Unparseable release dates: 0 of 135043


In [4]:
# 2. Is AppID unique and always an integer? Our whole schema's
# primary key depends on this being true.
print("Duplicate AppIDs:", df["AppID"].duplicated().sum())
print("AppID dtype:", df["AppID"].dtype)

Duplicate AppIDs: 0
AppID dtype: int64


In [5]:
# 3. Is Price always numeric and non-negative?
print("Price dtype:", df["Price"].dtype)
print("Negative prices:", (df["Price"] < 0).sum())
print("Missing prices:", df["Price"].isna().sum())

Price dtype: float64
Negative prices: 0
Missing prices: 0


In [6]:
# 4. Spot-check: do name/date/price/genre all look like they
# plausibly belong to the same game, across a random sample?
df.sample(10, random_state=1)[["Name", "Release date", "Price", "Genres"]]

,Name,Release date,Price,Genres
109723,Blacksmith Simulator,"Nov 28, 2025",5.99,"Adventure, Casual, Indie, Simulation"
26780,"Ethyrial, Echoes of Yore Playtest","Jan 24, 2025",0.00,NaN
78923,ETERNAL EXILE: BENEATH THE DARKNESS,"Aug 20, 2025",15.99,"Action, Indie"
51423,Where is my car? Playtest,"Sep 30, 2025",0.00,NaN
125316,PengPong: Prologue,"Feb 20, 2026",0.00,"Action, Casual, Indie, Free To Play"
88912,Stagehand Survival Simulator Playtest,"Feb 13, 2023",0.00,NaN
90366,Smash and Bash Monsters: Prologue,"Jul 22, 2024",0.00,"Action, Casual, Indie, RPG, Free To Play"
127770,Chronicles: Immortal Sect,"Mar 28, 2026",0.00,"Adventure, Massively Multiplayer, RPG, Free To..."
45862,Video World,"Jan 21, 2021",0.00,Indie
33603,Prince Of Wallachia,"Apr 28, 2021",0.59,"Action, Adventure, Indie"


In [7]:
# 5. total_reviews / review_score_percent sanity check
print(df[["total_reviews", "review_score_percent"]].describe())

       total_reviews  review_score_percent
count   1.350430e+05          82979.000000
mean    1.102677e+03             75.824456
std     3.110025e+04             23.868394
min     0.000000e+00              0.000000
25%     0.000000e+00             64.964021
50%     4.000000e+00             81.818182
75%     3.900000e+01             94.444444
max     8.815087e+06            100.000000


## Cohort selection
If everything above looks clean, select the actual research cohort: 2019–2022 releases with 500+ reviews.

In [8]:
candidates = loader.select_cohort(
    release_start="2019-01-01",
    release_end="2022-12-31",
    min_reviews=500,
    sample_size=260,  # buffer above 200, since some candidates will
                      # fail SteamCharts scraping in notebook 01
)
print(candidates.shape)
candidates[["AppID", "Name", "Release date", "total_reviews"]].head(10)

(260, 9)


,AppID,Name,Release date,total_reviews
0,1174390,ANNIE:Last Hope,"Apr 6, 2020",618
1,1893620,Circadian Dice,"Jul 11, 2022",578
2,1057090,Ori and the Will of the Wisps,"Mar 10, 2020",136690
3,1035560,Struggling,"Aug 27, 2020",681
4,1946550,Toilet Chronicles,"Jul 14, 2022",1389
5,1000010,Crown Trick,"Oct 16, 2020",5263
6,1879330,WARRIORS OROCHI 3 Ultimate Definitive Edition,"Jul 12, 2022",2260
7,740130,Tales of Arise,"Sep 9, 2021",34782
8,737520,Flynn: Son of Crimson,"Sep 15, 2021",676
9,1731520,Ark Mobius: Censored Edition,"Oct 14, 2021",995


In [9]:
candidates.to_csv("candidate_appids.csv", index=False)
print(f"Saved {len(candidates)} candidates to candidate_appids.csv")

Saved 260 candidates to candidate_appids.csv


In [10]:
# How does total_reviews (something we DO have for the full 5000+
# qualifying pool, before the 260-game random sample) compare between
# our final cohort and the full qualifying candidate pool it was
# drawn from? If they look similar, the sample is representative;
# if our cohort skews notably lower, that's evidence of real bias
# introduced somewhere in the sampling step itself.
full_qualifying_pool = loader.select_cohort(
    release_start="2019-01-01",
    release_end="2022-12-31",
    min_reviews=500,
    sample_size=None,  # no sampling — get the FULL qualifying pool
)
print("Full qualifying pool size:", len(full_qualifying_pool))
print(full_qualifying_pool["total_reviews"].describe())
print()
print("Our sampled cohort:")
print(candidates["total_reviews"].describe())


Full qualifying pool size: 3373
count    3.373000e+03
mean     1.139381e+04
std      4.885610e+04
min      5.000000e+02
25%      8.720000e+02
50%      1.789000e+03
75%      5.245000e+03
max      1.056677e+06
Name: total_reviews, dtype: float64

Our sampled cohort:
count       260.000000
mean      14780.257692
std       72140.382101
min         502.000000
25%         808.250000
50%        1619.000000
75%        5183.500000
max      994979.000000
Name: total_reviews, dtype: float64


In [11]:
release_dates = pd.to_datetime(loader.df["Release date"], errors="coerce")
mask = (
    (release_dates >= "2019-01-01") & (release_dates <= "2022-12-31")
    & (loader.df["total_reviews"] >= 500)
)
qualifying_before_genre_filter = loader.df[mask]

def is_real_game(genres_str):
    if pd.isna(genres_str):
        return False
    game_genres = {g.strip() for g in str(genres_str).split(",")}
    has_gameplay_genre = len(game_genres & KaggleLoader._REAL_GAME_GENRES) > 0
    has_non_game_tag = len(game_genres & KaggleLoader._NON_GAME_CATEGORIES) > 0
    return has_gameplay_genre and not has_non_game_tag

dropped = qualifying_before_genre_filter[
    ~qualifying_before_genre_filter["Genres"].apply(is_real_game)
]
print(f"{len(dropped)} games dropped out of {len(qualifying_before_genre_filter)} qualifying")

# Specifically isolate ones dropped ONLY because they lack a gameplay
# genre, not because they carry a giveaway non-game tag — these are
# the real, quantifiable false-negative risk
lacks_gameplay_only = dropped[dropped["Genres"].apply(
    lambda g: not pd.isna(g) and
    len(set(str(g).split(", ")) & KaggleLoader._NON_GAME_CATEGORIES) == 0
)]
print(f"Of those, {len(lacks_gameplay_only)} have no obvious non-game tag at all")
lacks_gameplay_only[["Name", "Genres"]].head(20)

46 games dropped out of 3419 qualifying
Of those, 2 have no obvious non-game tag at all


,Name,Genres
3360,Easy Pose,Early Access
81382,Intruder,Early Access
